# 02 - The Efficient Frontier

Sweep the risk-aversion parameter $\gamma$ to trace the unconstrained and long-only efficient frontiers for an $n$-asset universe.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from markowitz import MeanVariance

rng = np.random.default_rng(2024)
n = 8

mu_true = rng.uniform(0.03, 0.12, size=n)
A = rng.standard_normal((n, n))
Sigma_true = A @ A.T / n + 0.04 * np.eye(n)


## Sweep $\gamma$

Smaller $\gamma$ means more risk-tolerant: weights tilt toward the highest-Sharpe assets and the portfolio sits further along the frontier.

In [ ]:
gammas = np.geomspace(0.5, 50.0, 40)
frontier = []
for g in gammas:
    res = MeanVariance(risk_aversion=g, long_only=False).fit(mu_true, Sigma_true)
    w = res.weights_
    frontier.append((np.sqrt(w @ Sigma_true @ w), w @ mu_true))
frontier = np.array(frontier)


## Repeat with a long-only constraint

In [ ]:
frontier_lo = []
for g in gammas:
    res = MeanVariance(risk_aversion=g, long_only=True).fit(mu_true, Sigma_true)
    w = res.weights_
    frontier_lo.append((np.sqrt(w @ Sigma_true @ w), w @ mu_true))
frontier_lo = np.array(frontier_lo)


## Plot both frontiers

The long-only frontier lies inside the unconstrained one - constraints can only ever shrink the feasible set.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(frontier[:, 0], frontier[:, 1], lw=2, label='unconstrained')
ax.plot(frontier_lo[:, 0], frontier_lo[:, 1], lw=2, ls='--', label='long-only')
sig_assets = np.sqrt(np.diag(Sigma_true))
ax.scatter(sig_assets, mu_true, c='black', zorder=3, s=20)
ax.set_xlabel('portfolio standard deviation')
ax.set_ylabel('portfolio expected return')
ax.set_title('Efficient frontier (n = 8 assets)')
ax.legend()
fig.tight_layout()


## What just happened?

Every point on the curve is the solution of a separate QP - the library reused the same QP backend with different $\gamma$ values. In the next notebook we'll see what happens when we replace the *known* $\mu$ and $\Sigma$ with their **sample** estimates.